# The Augmented LLM

A **raw LLM call** is an isolated text predictor: you send a prompt, you get text back. The model only knows what was in its training data and whatever you fit in the context window for that single request.

An **augmented LLM** wraps the same core model with three engineering layers:

| Augmentation | What it adds | Without it |
|--------------|--------------|------------|
| **Retrieval** | Fresh facts from your docs, database, or index | Stale or hallucinated answers |
| **Tools** | Structured actions on the real world | Text-only advice that cannot execute |
| **Memory** | State that persists across turns | Every request starts from zero |

Augmentation turns a **chatbot** into an **assistant** that can **know** (retrieve), **do** (tools), and **remember** (memory). Add a **control loop** on top — decide when to retrieve, call tools, read memory, and stop — and you have an **agent**.

This notebook defines the augmented stack, compares it to bare completions, and builds a working **CloudMetrics API Developer Assistant** that uses all three augmentations end to end. Everything is self-contained — no prior notebooks or frameworks required.

In [ ]:
"""
Shared environment for the CloudMetrics API Developer Assistant.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- Semantic layer: API documentation chunks (retrieval source) ---
API_DOCS: list[dict[str, str]] = [
    {
        "id": "auth-01",
        "title": "Authentication",
        "text": "All API requests require an API key in the X-API-Key header. Keys are scoped per workspace. Rotate keys from Settings > Security.",
    },
    {
        "id": "metrics-02",
        "title": "Querying metrics",
        "text": "POST /v1/metrics/query accepts PromQL-like expressions. Rate limit: 100 requests/minute on Starter plan.",
    },
    {
        "id": "alerts-03",
        "title": "Alert webhooks",
        "text": "Alerts can POST to HTTPS endpoints. Payload includes alert_id, severity, and metric snapshot. Retry 3 times with exponential backoff.",
    },
    {
        "id": "sdk-04",
        "title": "Python SDK",
        "text": "pip install cloudmetrics. Initialize with CloudMetrics(api_key=os.environ['CM_API_KEY']). Use client.query('cpu_usage{host=\"web-1\"}').",
    },
    {
        "id": "billing-05",
        "title": "Plans and limits",
        "text": "Starter: 3 users, 30-day retention. Pro: unlimited users, 13-month retention, SSO. Enterprise: custom SLA and dedicated support.",
    },
]

# --- Simulated user profile store (memory target) ---
USER_PROFILES: dict[str, dict[str, str]] = {
    "dev-42": {
        "workspace": "acme-staging",
        "plan": "pro",
        "preferred_sdk": "python",
        "last_topic": "alert webhooks",
    },
}

SUPPORT_TICKETS: list[dict[str, Any]] = []

print(f"API docs loaded: {len(API_DOCS)} chunks")
print(f"User profiles: {len(USER_PROFILES)}")

---

## 1. Definition — LLM + Retrieval + Tools + Memory

| Layer | Purpose | CloudMetrics example |
|-------|---------|----------------------|
| **LLM** | Reasoning and natural language | Decide whether to search docs or open a ticket |
| **Retrieval** | Inject fresh, private facts | Pull `auth-01` for API key questions |
| **Tools** | Side-effecting actions | `search_docs`, `create_support_ticket` |
| **Memory** | Cross-turn and cross-session state | Remember user's workspace and preferred SDK |

The LLM is the **reasoning core**. The augmentations are **engineering you own** — retrieval quality, tool permissions, memory governance, and prompt assembly all live in your code, not in the model weights.

---

## 2. Architecture Diagram

```
                    ┌─────────────────────────────────────┐
  user message ────►│           Augmented LLM              │
                    │  ┌─────────┐  ┌───────┐  ┌───────┐  │
                    │  │ Memory  │  │  RAG  │  │ Tools │  │
                    │  │ thread  │  │ index │  │registry│  │
                    │  └───┬─────┘  └───┬───┘  └───┬───┘  │
                    │      └────────────┼──────────┘      │
                    │                   ▼                   │
                    │            Chat model core             │
                    └───────────────────┬───────────────────┘
                                        ▼
                              grounded answer / tool plan
```

Data flows **inward** before the model call (memory + retrieval enrich the prompt) and **outward** after (tools execute and results feed the next turn).

---

## 3. Raw ChatCompletion vs Augmented Assistant

| Dimension | Raw `chat.completions` | Augmented assistant |
|-----------|------------------------|---------------------|
| **Knowledge** | Frozen at training cutoff | Retrieval injects current docs |
| **Actions** | Text only unless you parse it | Native structured `tool_calls` |
| **Session** | Manual message list you manage | Memory layer with selective persistence |
| **Grounding** | Model may hallucinate policies | Citations from retrieved chunks |
| **Production fit** | Simple Q&A, drafting | Assistants, agents, workflows |

Augmentation adds **engineering surface area** — and responsibility. You own retrieval failures, tool auth, and memory conflicts.

In [ ]:
RAW_MESSAGES = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "How do I authenticate to CloudMetrics API?"},
]

retrieved = [d for d in API_DOCS if "auth" in d["id"]]
user_profile = USER_PROFILES["dev-42"]

AUGMENTED_MESSAGES = [
    {
        "role": "system",
        "content": (
            "You are a CloudMetrics API assistant. Answer using retrieved chunks. "
            "Cite chunk ids. User context: "
            f"workspace={user_profile['workspace']}, plan={user_profile['plan']}."
        ),
    },
    {
        "role": "user",
        "content": (
            "How do I authenticate to CloudMetrics API?\n\n"
            f"Retrieved chunks:\n{pretty(retrieved)}"
        ),
    },
]

print("Raw prompt:")
print(f"  Messages: {len(RAW_MESSAGES)}")
print(f"  System prompt chars: {len(RAW_MESSAGES[0]['content'])}")
print()
print("Augmented prompt:")
print(f"  Messages: {len(AUGMENTED_MESSAGES)}")
print(f"  System prompt chars: {len(AUGMENTED_MESSAGES[0]['content'])}")
print(f"  Extra context from retrieval + memory: yes")

---

## 4. Retrieval Component — Grounding in Private Knowledge

**Retrieval-Augmented Generation (RAG)** attaches a search index over your documents. The model sees only the top-k relevant snippets — not your entire wiki.

Pipeline:

1. User asks a question.
2. Retriever searches the index (keyword, vector, or hybrid).
3. Top chunks are injected into the prompt.
4. Model answers with citations.

Production systems use vector databases (FAISS, pgvector, Pinecone). For teaching, keyword overlap over a small doc set is enough to show the control flow.

In [ ]:
def retrieve_docs(query: str, top_k: int = 2) -> list[dict[str, str]]:
    """Keyword retriever over API_DOCS — swap for embeddings in production."""
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in API_DOCS:
        haystack = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in haystack) if terms else 0
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda x: (-x[0], x[1]["id"]))
    return [{"id": d["id"], "title": d["title"], "text": d["text"]} for _, d in scored[:top_k]]


def format_citations(chunks: list[dict[str, str]]) -> str:
    return "\n".join(f"[{c['id']}] {c['title']}: {c['text']}" for c in chunks)


queries = [
    "How do I authenticate with API key?",
    "Python SDK installation",
    "alert webhook retry policy",
]
for q in queries:
    hits = retrieve_docs(q)
    print(f"Q: {q}")
    print(f"  → [{hits[0]['id']}] {hits[0]['title']}" if hits else "  → no match")
    print()

---

## 5. Tools Component — From Language to Action

Tools let the model request **structured actions** instead of hallucinating shell commands or API calls.

The model emits `{name, arguments}`; your runtime validates and executes. Common tool categories:

| Tool type | Example | Risk level |
|-----------|---------|------------|
| **Read** | `search_docs`, `get_user_profile` | Low |
| **Write** | `create_support_ticket`, `update_alert` | Medium |
| **Execute** | `run_query`, `send_webhook_test` | Higher — needs auth + limits |

In [ ]:
def search_docs_tool(query: str) -> str:
    return format_citations(retrieve_docs(query, top_k=2))


def get_user_profile(user_id: str) -> dict[str, str]:
    return USER_PROFILES.get(user_id, {"error": "user not found"})


def create_support_ticket(user_id: str, subject: str, body: str) -> dict[str, Any]:
    ticket = {
        "id": f"TKT-{uuid.uuid4().hex[:6].upper()}",
        "user_id": user_id,
        "subject": subject,
        "body": body,
        "status": "open",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    SUPPORT_TICKETS.append(ticket)
    return ticket


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_docs": search_docs_tool,
    "get_user_profile": get_user_profile,
    "create_support_ticket": create_support_ticket,
}

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Search CloudMetrics API documentation.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_support_ticket",
            "description": "Open a support ticket for the user.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["user_id", "subject", "body"],
            },
        },
    },
]

print("Tools registered:", list(TOOL_REGISTRY.keys()))
print("\nDemo tool call:")
print(search_docs_tool("metrics query rate limit")[:120], "...")

---

## 6. Memory Component — Beyond the Context Window

Memory is **selective persistence** — not an infinitely long prompt.

| Memory type | What it holds | CloudMetrics example |
|-------------|---------------|----------------------|
| **Working** | Current conversation messages | Last user question + tool results |
| **Episodic** | Past session summaries | "Last week user asked about webhooks" |
| **Semantic** | Long-term facts about the user | Workspace name, plan tier, SDK preference |
| **Procedural** | How to act — playbooks, policies | "Always search docs before creating tickets" |

Production memory uses databases, vector stores, and TTL policies. The class below models all four types in memory.

In [ ]:
@dataclass
class AugmentedMemory:
    """Four memory types used by the augmented assistant."""

    user_id: str
    working: list[dict[str, str]] = field(default_factory=list)
    episodic: list[str] = field(default_factory=lambda: [
        "2026-07-01: User asked about alert webhook retries — resolved with alerts-03.",
    ])
    semantic: dict[str, str] = field(default_factory=dict)
    procedural: list[str] = field(default_factory=lambda: [
        "Search documentation before answering factual API questions.",
        "Create a support ticket only if docs do not resolve the issue.",
        "Never log or repeat API keys in responses.",
    ])

    def __post_init__(self) -> None:
        if not self.semantic:
            profile = USER_PROFILES.get(self.user_id, {})
            self.semantic = dict(profile)

    def add_message(self, role: str, content: str) -> None:
        self.working.append({"role": role, "content": content})

    def remember_fact(self, key: str, value: str) -> None:
        self.semantic[key] = value
        self.episodic.append(f"stored semantic fact: {key}={value!r}")

    def context_block(self) -> str:
        """Assembled block injected into the system prompt."""
        lines = [
            "## User profile (semantic memory)",
            pretty(self.semantic),
            "## Recent sessions (episodic memory)",
            "\n".join(f"- {e}" for e in self.episodic[-2:]),
            "## Policies (procedural memory)",
            "\n".join(f"- {p}" for p in self.procedural),
        ]
        return "\n".join(lines)

    def snapshot(self) -> dict[str, Any]:
        return {
            "working_messages": len(self.working),
            "episodic_entries": len(self.episodic),
            "semantic_keys": list(self.semantic.keys()),
            "procedural_rules": len(self.procedural),
        }


memory = AugmentedMemory(user_id="dev-42")
memory.add_message("user", "I'm using the Python SDK on staging")
memory.remember_fact("last_topic", "python sdk staging")
print(memory.context_block()[:400], "...")
print("\nSnapshot:", memory.snapshot())

---

## 7. Stacking Augmentations — Order Matters

A safe default pipeline:

```
memory inject  →  retrieve  →  model (plan)  →  tools (optional)  →  model (finalize)
```

| Step | Why this order |
|------|----------------|
| **Memory first** | Personalizes retrieval queries and tool args |
| **Retrieve before answer** | Stops model from answering from parametric memory alone |
| **Tools after plan** | Model commits to structured call before side effects |
| **Finalize after tools** | Model synthesizes tool observations into natural language |

Retrieving **after** the model already answered defeats the purpose of RAG. Executing tools **before** the model plans wastes API calls.

In [ ]:
def augmentation_pipeline_steps(query: str, mem: AugmentedMemory) -> list[dict[str, str]]:
    """Trace each augmentation layer in order."""
    steps = []
    steps.append({"step": "1_memory_inject", "detail": mem.context_block()[:80] + "..."})
    chunks = retrieve_docs(query)
    steps.append({"step": "2_retrieve", "detail": f"{len(chunks)} chunks: {[c['id'] for c in chunks]}"})
    steps.append({"step": "3_model_plan", "detail": "decide: answer from chunks OR call tool"})
    if "ticket" in query.lower() or "support" in query.lower():
        steps.append({"step": "4_tool_execute", "detail": "create_support_ticket(...)"})
    else:
        steps.append({"step": "4_tool_skip", "detail": "no tool needed"})
    steps.append({"step": "5_model_finalize", "detail": "compose cited answer"})
    return steps


for step in augmentation_pipeline_steps("How do API keys work?", memory):
    print(f"{step['step']}: {step['detail']}")

---

## 8. The AugmentedLLM Class — End-to-End Integration

Below, all three augmentations are wired into one class. A **rule-based planner** stands in for the LLM so the notebook runs offline; in production you replace `plan()` with an API call that receives memory, retrieval, and tool schemas.

In [ ]:
@dataclass
class AugmentedLLM:
    """
    Augmented LLM = memory + retrieval + tools + planner.
    Add a loop with max_steps on top to make this an agent.
    """

    user_id: str
    memory: AugmentedMemory = field(init=False)
    trace: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.memory = AugmentedMemory(user_id=self.user_id)

    def _log(self, msg: str) -> None:
        self.trace.append(msg)

    def plan(self, query: str, chunks: list[dict[str, str]]) -> dict[str, Any]:
        """Offline planner — mimics LLM deciding answer vs tool call."""
        q = query.lower()
        if any(w in q for w in ("ticket", "support", "escalate", "not working")):
            return {
                "type": "tool_call",
                "name": "create_support_ticket",
                "arguments": {
                    "user_id": self.user_id,
                    "subject": query[:60],
                    "body": f"User question: {query}. Docs searched: {[c['id'] for c in chunks]}",
                },
            }
        return {"type": "text", "content": format_citations(chunks)}

    def invoke(self, query: str) -> str:
        self.trace = []
        self.memory.add_message("user", query)
        self._log("memory injected into context")

        chunks = retrieve_docs(query)
        self._log(f"retrieved {len(chunks)} chunks: {[c['id'] for c in chunks]}")

        decision = self.plan(query, chunks)
        self._log(f"model plan: {decision['type']}")

        if decision["type"] == "tool_call":
            name = decision["name"]
            args = decision["arguments"]
            result = TOOL_REGISTRY[name](**args)
            self._log(f"tool {name} → {result.get('id', result)}")
            self.memory.add_message("tool", pretty(result))
            answer = (
                f"I opened support ticket {result['id']} for you. "
                f"Meanwhile, from docs: {format_citations(chunks)[:200]}..."
            )
        else:
            answer = (
                f"Using your profile ({self.memory.semantic.get('workspace', 'unknown')}):\n"
                f"{decision['content']}"
            )

        self.memory.add_message("assistant", answer)
        self._log("model finalized response")
        return answer


assistant = AugmentedLLM(user_id="dev-42")

print("=" * 72)
print("QUERY 1: Factual (retrieval + memory)")
print(assistant.invoke("How do I authenticate to the API?"))
print("Trace:", assistant.trace)

print("\n" + "=" * 72)
print("QUERY 2: Action (tool + retrieval)")
assistant2 = AugmentedLLM(user_id="dev-42")
print(assistant2.invoke("Webhooks not working — please open a support ticket"))
print("Trace:", assistant2.trace)

---

## 9. From Augmented LLM to Agent

| | **Augmented LLM** | **Agent** |
|--|-------------------|-----------|
| **Structure** | One enriched model call (or short chain) | Repeated perceive → plan → act loop |
| **Tools** | May call zero or one tool per invoke | May call many tools across steps |
| **Stop condition** | Single response | Task done, max_steps, or guardrail |

An agent is an **augmented LLM plus a runtime loop**. The augmentations are the same; the agent adds iteration, guardrails, and termination logic.

In [ ]:
@dataclass
class AugmentedAgent(AugmentedLLM):
    """Extends AugmentedLLM with a multi-step loop — agent = augmented LLM + loop."""

    max_steps: int = 3

    def run(self, goal: str) -> str:
        """
        Multi-step loop: search → if not resolved → ticket.
        Demonstrates agent iteration over the same augmentations.
        """
        self.trace = []
        self._log(f"agent goal: {goal}")

        # Step 1: try retrieval answer
        chunks = retrieve_docs(goal)
        self._log(f"step 1 retrieve: {[c['id'] for c in chunks]}")

        if "not working" in goal.lower() or "broken" in goal.lower():
            # Step 2: escalate to tool
            ticket = create_support_ticket(
                self.user_id,
                subject=goal[:50],
                body=f"Auto-escalated after doc search: {[c['id'] for c in chunks]}",
            )
            self._log(f"step 2 tool: ticket {ticket['id']}")
            return f"Docs consulted ({chunks[0]['id']}). Issue persists — ticket {ticket['id']} opened."

        return f"Resolved from docs [{chunks[0]['id']}]: {chunks[0]['text'][:100]}..."


agent = AugmentedAgent(user_id="dev-42")
print(agent.run("alert webhooks not working after config change"))
print("Agent trace:", agent.trace)

---

## 10. In-Context Chunks vs Vector Store

| Approach | When to use | Trade-off |
|----------|-------------|-----------|
| **Inline chunks** (this notebook) | Teaching, tiny corpora (<50 docs) | Simple; does not scale |
| **Vector store** (FAISS, pgvector, Pinecone) | Production doc sets | Needs embedding pipeline + index maintenance |
| **Hosted file search** (OpenAI Assistants API) | Fast prototype with managed infra | Less control over chunking and ranking |

The retrieval **interface** stays the same — `retrieve(query) -> chunks` — whether you use keywords or embeddings behind it.

In [ ]:
class KeywordIndex:
    """Thin abstraction — swap internals for vectors without changing callers."""

    def __init__(self, docs: list[dict[str, str]]) -> None:
        self.docs = docs

    def search(self, query: str, top_k: int = 2) -> list[dict[str, str]]:
        return retrieve_docs(query, top_k)  # same logic, index-backed


index = KeywordIndex(API_DOCS)
print("Index search 'pro plan retention':")
for hit in index.search("pro plan retention"):
    print(f"  [{hit['id']}] {hit['title']}")

---

## 11. DIY Augmented Loop vs Hosted Assistants API

You can build augmentation yourself (this notebook) or use a hosted product that bundles pieces:

| Capability | DIY augmented loop | Hosted assistant (e.g. OpenAI Assistants) |
|------------|-------------------|-------------------------------------------|
| **Retrieval** | You own chunking + index | Managed vector stores / file search |
| **Tools** | Custom sandbox + auth | Built-in tools + your function schemas |
| **Memory** | Your database / state | Thread objects managed by provider |
| **Control** | Full prompt and pipeline control | Faster setup, less customization |

Hosted lifecycle (conceptual):

1. Create thread
2. Add user message
3. Create run (model + tools + file_search)
4. Poll until `requires_action` or `completed`
5. Submit tool outputs if needed
6. Retrieve final messages

Both approaches bottom out on the same primitive — a chat model call with enriched context.

---

## 12. Failure Modes Unique to Augmented LLMs

| Failure | Symptom | Mitigation |
|---------|---------|------------|
| **Bad retrieval** | Confident wrong answer | Re-rank, hybrid search, force retrieve step |
| **Skipped retrieval** | Model answers from training data | Require search tool before factual answers |
| **Tool misuse** | Wrong args, dangerous calls | Schema validation, allow-lists, approval gates |
| **Stale memory** | Outdated user preferences | TTL, version tags, conflict resolution |
| **Context overflow** | Lost early turns | Summarize working memory, trim chunks |
| **Augmentation order bug** | Tools run before retrieval | Enforce pipeline order in code |

In [ ]:
def diagnose_failure(symptom: str) -> str:
    mapping = {
        "wrong api version cited": "bad retrieval — refresh index, add re-ranker",
        "answered without searching": "skipped retrieval — force search_docs first",
        "ticket created for simple faq": "tool misuse — tighten planner / add intent check",
        "used old workspace name": "stale memory — add TTL to semantic facts",
    }
    return mapping.get(symptom, "check trace: memory → retrieve → plan → tool → finalize")


for symptom in ["answered without searching", "used old workspace name"]:
    print(f"{symptom} → {diagnose_failure(symptom)}")

---

## 13. Token and Context Comparison

Augmentation adds tokens. Measuring the delta helps you decide how many chunks and memory facts to inject.

In [ ]:
def estimate_prompt_chars(messages: list[dict[str, str]]) -> int:
    return sum(len(m["content"]) for m in messages)


raw_chars = estimate_prompt_chars(RAW_MESSAGES)
aug_chars = estimate_prompt_chars(AUGMENTED_MESSAGES)

print(f"Raw completion prompt:        {raw_chars:>6} chars")
print(f"Augmented prompt:             {aug_chars:>6} chars")
print(f"Augmentation overhead:        {aug_chars - raw_chars:>+6} chars")
print(f"\nRough token estimate (÷4):   {(aug_chars - raw_chars) // 4:>+6} tokens")

---

## 14. Multiple Augmented LLMs → Multi-Agent System

When each specialist is itself an **augmented LLM** (own retrieval scope, tools, and memory), coordination between them forms a **multi-agent system**:

```
Supervisor (light LLM)
    ├── Docs agent    (retrieval over API_DOCS + search tool)
    ├── Support agent (ticket tools + episodic memory)
    └── Billing agent (plan docs + read-only account tool)
```

**Augmentation is per-agent.** Orchestration (supervisor, handoffs, shared state) is **between** agents. Design each agent's augmentations narrowly — a billing agent does not need webhook docs in its index.

In [ ]:
SPECIALIST_AGENTS = {
    "docs_agent": {
        "retrieval_scope": ["auth-01", "metrics-02", "sdk-04"],
        "tools": ["search_docs"],
        "memory": "working + semantic",
    },
    "support_agent": {
        "retrieval_scope": ["alerts-03"],
        "tools": ["create_support_ticket", "get_user_profile"],
        "memory": "episodic + procedural",
    },
    "billing_agent": {
        "retrieval_scope": ["billing-05"],
        "tools": ["get_user_profile"],
        "memory": "semantic only",
    },
}

print(f"{'Agent':<16} {'Docs':<8} {'Tools':<30} Memory")
print("-" * 70)
for name, cfg in SPECIALIST_AGENTS.items():
    print(f"{name:<16} {len(cfg['retrieval_scope']):<8} {', '.join(cfg['tools']):<30} {cfg['memory']}")

---

## 15. Optional — Live LLM with Augmented Context

Set `USE_LIVE_LLM = True` with a valid API key. The call is still a raw `chat.completions` — the augmentation is in **what you put in the messages**.

In [ ]:
USE_LIVE_LLM = False

def build_augmented_prompt(query: str, user_id: str) -> list[dict[str, str]]:
    mem = AugmentedMemory(user_id=user_id)
    chunks = retrieve_docs(query)
    return [
        {"role": "system", "content": f"Assistant. Policies:\n{mem.context_block()}"},
        {"role": "user", "content": f"{query}\n\nDocs:\n{format_citations(chunks)}"},
    ]


OFFLINE_DEMO = build_augmented_prompt("How do API keys work?", "dev-42")
print(f"Augmented message bundle: {len(OFFLINE_DEMO)} messages, "
      f"{sum(len(m['content']) for m in OFFLINE_DEMO)} chars")

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        messages = build_augmented_prompt("How do API keys work?", "dev-42")
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            max_tokens=150,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Treating memory as infinite context | Token blow-up, lost signal | Selective recall + summarization |
| Retrieval after model already answered | Hallucination | Enforce retrieve-before-generate |
| Too many chunks in prompt | Noise, cost | Top-k + re-ranker |
| Tools without validation | Bad side effects | Schema + allow-list |
| Confusing augmented LLM with agent | Missing loop and guardrails | Add max_steps and termination |
| Same index for all specialists | Irrelevant retrieval | Per-agent retrieval scope |

---

## 17. Check Your Understanding

1. Name the three augmentations around the LLM core and what each adds.
2. What is the safe order for stacking memory, retrieval, model, and tools?
3. How does an **augmented LLM** differ from an **agent**?
4. Name the four memory types and give a CloudMetrics example for each.
5. Why does raw `chat.completions` struggle with "What is our current API rate limit for Pro?"
6. What changes when you give each specialist in a MAS its own retrieval scope?
7. What failure mode does "answered without searching" indicate and how do you fix it?

---

## 18. Summary

An **augmented LLM** is a core language model plus **retrieval** (know fresh facts), **tools** (act on the world), and **memory** (persist state across turns). It is the foundation under most modern assistants and agents.

**Key takeaways:**

- **Retrieval** grounds answers in your private docs — inject chunks *before* the model answers.
- **Tools** turn language into validated actions — the runtime executes, not the model.
- **Memory** spans working, episodic, semantic, and procedural layers — not just a long prompt.
- **Stacking order** matters: memory → retrieve → plan → tool → finalize.
- An **agent** adds a **control loop** (iteration + guardrails) on top of the same augmentations.
- **Multiple augmented LLMs** with narrow scopes become specialists in a multi-agent system.
- Whether you build DIY or use hosted assistants, you still own retrieval quality, tool auth, and memory governance.

You can now design any assistant by asking: *What must it know (retrieval)? What must it do (tools)? What must it remember (memory)? Does it need a loop (agent)?*